**Ingest Sprints - Incremental**

Reads all JSON files from the `sprints/` folder in the batch landing path, adds metadata, and writes to `formula1_incr.bronze.sprints` partitioned by `batch_id`.

**Load config and helpers**

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

**Set variables**

In [0]:
dbutils.widgets.text('p_batch_id','')
batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
source_file = f'{loding_folder_path}/{batch_id}/sprints'
table_name = f'{catalog_name}.{bronze_schema}.sprints' # to replace the save table in write api 

**Verify**

**Define schema** (StructType, 14 columns)

In [0]:
from pyspark.sql.types import *
sprints_schema = StructType([
    StructField('date', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('round', IntegerType(), True),
    StructField('season', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('constructorId', StringType(), True),
    StructField('driverId', StringType(), True),
     StructField('grid', IntegerType(), True),
    StructField('laps', IntegerType(), True),
     StructField('number', IntegerType(), True),
     StructField('points', FloatType(), True),
    StructField('position', IntegerType(), True),
    StructField('positionText', StringType(), True),
    StructField('status', StringType()),
   

])

**Read JSON folder** (multiLine option for multi-line JSON files)

In [0]:
sprints_df = (
    spark.read
     .format('json')
    # .option('Headers',True)
     .schema(sprints_schema)
     .option('multiLine', True)
     .load(source_file)
)


In [0]:
display(sprints_df)

**Add metadata columns**

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)


**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
write_to_bronze(
    input_df = sprints_final_df,
    table_name = table_name,
    batch_id = batch_id
)

In [0]:
display(spark.table(table_name))


In [0]:
%sql
select season, count(*)
from formula1_incr.bronze.sprints
group by season
order by season asc